# Google ADK on AgentCore Runtime demo

Demonstrates Google ADK agent hosted in Amazon Bedrock AgentCore Runtime.


## Tutorial step-by-step

#### Setup python libraries

In [ ]:
# Agentcore starter toolkit (deprecated) for deploying the agent
#%pip install -r requirements-dev.txt --quiet

# [Optional] Agent dependent libraries for local development
#%pip install -r requirements.txt --quiet

### Configure AgentCore Runtime for deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code. 

Please note that when using the `bedrock_agentcore_starter_toolkit` to configure your agent, it takes care of the opentelemetry instrumentation. 

##### Add Gemini API key
agentcore identity add-credential-provider \
  --name google \
  --type API_KEY \
  --api-key "YOUR_GEMINI_API_KEY"

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()
agent_name = "google_adk_demo"
response = agentcore_runtime.configure(
    entrypoint="googleoinftravelctr/main.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region="us-east-1",
    agent_name=agent_name,
    memory_mode='NO_MEMORY',
)
response

### Deploy to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

In [ ]:
launch_result = agentcore_runtime.launch(
    env_vars={
        "AWS_REGION": "us-east-1",
        # "AGENT_OBSERVABILITY_ENABLED": "true",
        # "AWS_GENAI_CONTENT_EXTRACTION_OPT_OUT": "true"
        
    }
)
launch_result

### Check Deployment Status

Wait for the runtime and memory to be ready before invoking:

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload. Invoke multiple times.

In [ ]:
# !agentcore invoke --help
!agentcore invoke --user-id demo-user --session-id "testing-google-adk-$(date +%Y%m%d-%H%M%S)" '{"prompt": "search flights from seattle to NY trip for jul 4th 2026"}'


## Cleanup instructions

Don't forget to cleanup any resources created. Remove `--dry-run` for actual deletion.

In [ ]:
!agentcore destroy --delete-ecr-repo --force --dry-run